# `candidate_feature_set_trimming`

## Purpose

Grow candidate feature sets by iteratively adding features that significantly improve performance.

## Previous notebook

`iterative_feature_set_growth`

## Next notebook

`candidate_feature_set_trimming`

# Imports

In [1]:
import numpy as np
import pandas as pd

# import random

import os
from datetime import datetime
import pickle

import random
import time

import warnings

# from skimpy import skim

import matplotlib.pyplot as plt
import seaborn as sns

from statistics import median

from scipy.stats import ttest_rel, t

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, average_precision_score, RocCurveDisplay, accuracy_score

import prepare_data

original_dir = os.getcwd()
os.chdir('../jupyter')
# import load_data
# import feature_name_functions
import cutpoint_analysis
os.chdir(original_dir)

warnings.filterwarnings("ignore")

# Load data

In [2]:
train_cohort_ll = prepare_data.load_and_process_cohort('train', 'latest')
train_cohort_med = prepare_data.load_and_process_cohort('train', 'median')

# Define functions to use to trim feature sets

## Cross-validation on input feature sets

In [3]:
def train_logreg_cv_from_feature_set(data_df, 
                                     feature_set,
                                     scoring_metric,
                                     max_iter=10000,
                                     n_splits=10,
                                     n_jobs=10,
                                     random_state=343):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = LogisticRegression(class_weight='balanced',
                               max_iter=max_iter)
    
    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

def train_svc_cv_from_feature_set(data_df, 
                                 feature_set, 
                                 scoring_metric,
                                 n_splits=10,
                                 n_jobs=10,
                                 random_state=343,
                                 max_iter=2500,
                                 svc_sample_proportion=1.0,
                                 svc_sample_random_state=343):
    if svc_sample_proportion < 1.0:
        total_dataset_size = len(data_df)
        samples_to_use = int(np.round(svc_sample_proportion * total_dataset_size))
        positive_class_size = len(data_df[data_df['aki_72hrs_any']==1])
        positive_samples_to_use = int(np.round((positive_class_size / total_dataset_size) * samples_to_use))
        negative_samples_to_use = samples_to_use - positive_samples_to_use
        positive_samples = data_df[data_df['aki_72hrs_any']==1].sample(n = positive_samples_to_use, random_state=svc_sample_random_state)
        negative_samples = data_df[data_df['aki_72hrs_any']==0].sample(n = negative_samples_to_use, random_state=svc_sample_random_state)
        svc_sample = pd.concat([positive_samples, negative_samples])
    else:
        svc_sample = data_df
        
    X = svc_sample[feature_set].to_numpy()
    y = svc_sample['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = SVC(class_weight='balanced',
                max_iter=max_iter,
                probability=True)
    
    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

## Hypothesis test to compare cross-validation scores

## Iterative trimming of candidate set

In [4]:
def get_critical_value(sig_level, dof):
    return t.ppf(0.5 - sig_level, dof)

def get_t_value_paired_test(value_list_1, value_list_2, verbosity=1):
    if len(value_list_1) == len(value_list_2):
        diffs = [value_list_2[i] - value_list_1[i] for i in range(len(value_list_1))]
        diffs_mean = np.mean(diffs)
        diffs_sd = np.sqrt(sum([(diff - diffs_mean)**2 for diff in diffs]) / (len(diffs) + 1))
        t_val = diffs_mean / (diffs_sd / np.sqrt(len(diffs)))
        if verbosity > 0:
            print('mean of diffs = %.4f' % diffs_mean)
            print('sd of diffs = %.4f' % diffs_sd)
            print('t = %.4f' % t_val)
        return t_val
    else:
        print('Paired sample t-test requires input value lists to be the same size. Returning None.')
        return None
    
def check_if_new_scores_not_worse(original_scores, new_scores, sig_level):
    critical_val = get_critical_value(sig_level, len(original_scores) - 1)
    t_val = get_t_value_paired_test(original_scores, new_scores)
    new_scores_not_worse = (t_val > critical_val)
    return new_scores_not_worse

def iteratively_trim_feature_set(
    df,
    candidate_feature_set,
    scoring_metric, # 'roc_auc' or 'average_precision'
    imp_method_str,
    random_state_list = [343, 0, 42],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=[],
    original_feature_set_svc_scores=[],
    max_iter_svc=-1,
    allow_overwriting_files=False,
    subfolder_name='',
    logreg_only=False,
    svc_sample_proportion=1.0,
    svc_sample_random_state=343
):
    # Make sure directory structure is set up
    # results_dir = 'pickle/trimmed_feature_sets/' + \
    #     subfolder_name + ('/' if len(subfolder_name) > 0 else '') + \
    #         imp_method_str + '_' + scoring_metric + '_feature_dicts'# + \
    #             # 'random_state_' + str(random_state)
    results_dir_path_folder_list = [
        'pickle',
        'trimmed_feature_sets',
        subfolder_name + ('/' if len(subfolder_name) > 0 else ''),
        imp_method_str + '_' + scoring_metric + '_feature_dicts'
    ]
    
    temp_path_str = '.'
    
    for subpath_str in results_dir_path_folder_list:
        temp_path_str += ('/' + subpath_str)
        if os.path.isdir(temp_path_str) == False:
            os.mkdir(temp_path_str)
            
    for random_state in random_state_list:
        if os.path.isdir(temp_path_str + '/random_state_%d' % random_state) == False:
            os.mkdir(temp_path_str + '/random_state_%d' % random_state)
            
    # Get reduced SVC training set if desired
    if svc_sample_proportion < 1.0:
        total_dataset_size = len(df)
        samples_to_use = int(np.round(svc_sample_proportion * total_dataset_size))
        positive_class_size = len(df[df['aki_72hrs_any']==1])
        positive_samples_to_use = int(np.round((positive_class_size / total_dataset_size) * samples_to_use))
        negative_samples_to_use = samples_to_use - positive_samples_to_use
        positive_samples = df[df['aki_72hrs_any']==1].sample(n = positive_samples_to_use, random_state=svc_sample_random_state)
        negative_samples = df[df['aki_72hrs_any']==0].sample(n = negative_samples_to_use, random_state=svc_sample_random_state)
        svc_training_sample = pd.concat([positive_samples, negative_samples])
    else:
        svc_training_sample = df.copy()
    
    # Determine scores for full candidate feature set. These will be compared to the performance
    # of models trained on subsets of our candidate features to see which features can be 
    # removed without a significant drop in performance.

    # Logisitic regression
    if len(original_feature_set_logreg_scores) == 0:
        print('Getting initial scores for full feature set on logistic regression model.')
        original_feature_set_logreg_scores = train_logreg_cv_from_feature_set(
            df, 
            candidate_feature_set, 
            scoring_metric=scoring_metric,
            n_splits=10,
            n_jobs=10,
            random_state=random_state)
    
        print('Finished training logistic regression model on full feature set.')
        print('Mean logistic regression ' + scoring_metric + ' score: %.3f' % np.mean(original_feature_set_logreg_scores))

    # SVC
    if len(original_feature_set_svc_scores) == 0 and logreg_only == False:
        print('Getting initial scores for full feature set on SVC model.')
        original_feature_set_logreg_scores = train_svc_cv_from_feature_set(
            svc_training_sample, 
            candidate_feature_set, 
            scoring_metric=scoring_metric,
            n_splits=10,
            n_jobs=10,
            random_state=random_state,
            max_iter=max_iter_svc)
    
        print('Finished training SVC model on full feature set.')
        print('Mean SVC ' + scoring_metric + ' score: %.3f' % np.mean(original_feature_set_svc_scores))
    
    n_features = len(candidate_feature_set)
    print('Starting iteration.')
    start_time = time.time()
    
    n = 0
    n_loops = len(random_state_list)
    
    for random_state in random_state_list:
        best_feature_set_logreg_scores = original_feature_set_logreg_scores.copy()
        best_feature_set_svc_scores = original_feature_set_svc_scores.copy()
        random.seed(random_state)
        n += 1
        
        # Initialize temp_svc_scores to None so we don't get an error when we define temp_dict
        temp_svc_scores = None
        
        loop_start_time = time.time()
        elapsed_seconds_total = time.time() - start_time
        print('Starting loop %d after %.2f minutes.' % (n, elapsed_seconds_total / 60))
        
        random.shuffle(candidate_feature_set)
        current_feature_set = candidate_feature_set.copy()
    
        feature_number = 0

        # For each feature, compare performance of model trained on feature set with that feature 
        # removed to performance on entire candidate feature set. If performance is not significantly
        # reduced, remove that feature from feature set.
        for feature in candidate_feature_set:
            temp_filename = 'pickle/trimmed_feature_sets/' + \
                subfolder_name + ('/' if len(subfolder_name) > 0 else '') + \
                    imp_method_str + '_' + scoring_metric + '_feature_dicts/' + \
                        'random_state_' + str(random_state) + '/' + \
                            feature + '.pickle'
            
            if os.path.isfile(temp_filename) == False or allow_overwriting_files == True:
                feature_number += 1
                print('~'*20)
                print(feature + ' (loop %d of %d, feature %d of %d)' % (n, n_loops, feature_number, n_features))
                print('Total elapsed time: %.2f minutes.' % ((time.time() - start_time) / 60))
                print('Loop elapsed time: %.2f minutes.' % ((time.time() - loop_start_time) / 60))

                # Get logistic regression scores first (SVC training time is much longer, and if
                # logistic regression score drops significantly when feature is removed, we know
                # that we'll keep it in the feature set without having to test SVC.
                temp_feature_set = current_feature_set.copy()
                temp_feature_set.remove(feature)
                temp_logreg_scores = train_logreg_cv_from_feature_set(
                    df, 
                    temp_feature_set,
                    random_state=random_state,
                    scoring_metric=scoring_metric
                )
                # log_stat, log_p = cv_auc_hypothesis_test(best_feature_set_logreg_scores, temp_logreg_scores, 'less')
                print('Original mean logreg AUC: %.4f' % np.mean(best_feature_set_logreg_scores))
                print('Temp mean logreg AUC: %.4f' % np.mean(temp_logreg_scores))

                
                new_scores_not_worse_logreg = check_if_new_scores_not_worse(
                    best_feature_set_logreg_scores, temp_logreg_scores, pval_threshold
                )
                if new_scores_not_worse_logreg:#log_p > pval_threshold:# or log_stat < 0:#  or np.mean(temp_logreg_scores) > np.mean(full_feature_set_logreg_scores):
                    print('Logistic regression model not significantly worse after removing ' + feature + '.')
                    print('Logistic regression scores (new | old | new - old):')
                    for i in range(len(temp_logreg_scores)):
                        print('%.4f | %.4f | %.4f' % (
                            temp_logreg_scores[i], best_feature_set_logreg_scores[i], temp_logreg_scores[i] - best_feature_set_logreg_scores[i]
                        ))

                    # svc_stat, svc_p = cv_auc_hypothesis_test(best_feature_set_svc_scores, temp_svc_scores, 'less')
                    
                    if np.mean(temp_logreg_scores) > np.mean(best_feature_set_logreg_scores):
                        best_feature_set_logreg_scores = temp_logreg_scores.copy()
                        
                    if logreg_only == False:
                        # If logistic regression performance does not drop significantly, get CV scores
                        # for SVC
                        print('\n' + 'Assessing SVC performance...')

                        temp_svc_scores = train_svc_cv_from_feature_set(
                            svc_training_sample,
                            temp_feature_set,
                            random_state=random_state,
                            scoring_metric=scoring_metric,
                            max_iter=max_iter_svc
                        )

                        print('Original mean SVC AUC: %.4f' % np.mean(best_feature_set_svc_scores))
                        print('Temp mean SVC AUC: %.4f' % np.mean(temp_svc_scores))
                        
                        # If SVC performance is also not significantly worse, remove feature from
                        # feature set.
                        new_scores_not_worse_svc = check_if_new_scores_not_worse(
                            best_feature_set_svc_scores, temp_svc_scores, pval_threshold
                        )
                        if new_scores_not_worse_svc:#svc_p > pval_threshold:# or svc_stat < 0:# np.mean(temp_svc_scores) > np.mean(full_feature_set_svc_scores):
                            print('SVC model also not significantly worse after removing ' + feature + '.')
                            print('SVC scores (new | old | new - old):')
                            for i in range(len(temp_svc_scores)):
                                print('%.4f | %.4f | %.4f' % (
                                    temp_svc_scores[i], best_feature_set_svc_scores[i], (temp_svc_scores[i] - best_feature_set_svc_scores[i])
                                ))

                            print('\n' + 'Removing ' + feature + ' from current feature set.')
                            current_feature_set = temp_feature_set.copy()
                            print('Number of features in current feature set: %d.' % len(current_feature_set))

                            if np.mean(temp_svc_scores) > np.mean(best_feature_set_svc_scores):
                                best_feature_set_svc_scores = temp_svc_scores.copy()
                    else:
                        print('\n' + 'Removing ' + feature + ' from current feature set.')
                        current_feature_set = temp_feature_set.copy()

                print('~'*20)

                temp_dict = {
                    'imp': 'latest lab',
                    'metric': 'auroc',
                    'feature': feature,
                    'improved_logreg': 1 if np.mean(temp_logreg_scores) > np.mean(best_feature_set_logreg_scores) else 0,
                    'logreg_scores_when_removed': temp_logreg_scores,
                    'improved_svc': 1 if ((temp_svc_scores is not None) and (np.mean(temp_svc_scores) > np.mean(best_feature_set_svc_scores))) else 0,
                    'svc_scores_when_removed': temp_svc_scores
                }
                with open(temp_filename, 'wb') as outfile:
                    pickle.dump(temp_dict, outfile)
    
        final_feature_set_list.append(current_feature_set)

    return final_feature_set_list

## Load initial CV scores if available, otherwise calculate

In [5]:
def get_or_load_initial_scores(
    train_cohort,
    candidate_feature_set,
    scoring_metric,
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/v2/ll_v2_roc_auc_feature_dicts/all_candidate_features_logreg_scores.pickle',
    get_svc_scores=True,
    svc_sample_proportion=1.0,
    svc_sample_random_state=343
):
    if os.path.isfile(logreg_score_file_path):
        print('Loading full logistic regression scores...')
        with open(logreg_score_file_path, 'rb') as infile:
            logreg_scores = pickle.load(infile)
    else:
        print('Calculating full logistic regression scores...')
        logreg_scores = train_logreg_cv_from_feature_set(
            train_cohort, 
            candidate_feature_set, 
            scoring_metric=scoring_metric,
            n_splits=n_splits,
            n_jobs=n_jobs,
            random_state=random_state
        )

        with open(logreg_score_file_path, 'wb') as outfile:
            pickle.dump(logreg_scores, outfile)

    if get_svc_scores:
        svc_score_file_path = logreg_score_file_path.replace('logreg', 'svc')
        if os.path.isfile(svc_score_file_path):
            print('Loading full SVC scores...')
            with open(svc_score_file_path, 'rb') as infile:
                svc_scores = pickle.load(infile)
        else:
            print('Calculating full SVC scores...')
            svc_scores = train_svc_cv_from_feature_set(
                train_cohort, 
                candidate_feature_set, 
                scoring_metric=scoring_metric,
                n_splits=n_splits,
                n_jobs=n_jobs,
                random_state=random_state,
                max_iter=max_iter_svc,
                svc_sample_proportion=svc_sample_proportion,
                svc_sample_random_state=svc_sample_random_state
            )

            with open(svc_score_file_path, 'wb') as outfile:
                pickle.dump(svc_scores, outfile)
                
        return logreg_scores, svc_scores
                
    else:
        return logreg_scores

# Trim feature sets

## Latest lab imputation

### AUROC

In [6]:
with open('pickle/candidate_features/svc_latest_lab_auroc_candidate_features.pickle', 'rb') as infile:
    logreg_ll_roc_candidate_set = pickle.load(infile)

In [7]:
full_auroc_ll_set_logreg_scores, full_auroc_ll_set_svc_scores = get_or_load_initial_scores(
    train_cohort_ll,
    logreg_ll_roc_candidate_set,
    'roc_auc',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/ll/auroc.pickle',
    get_svc_scores=True
)

Loading full logistic regression scores...
Loading full SVC scores...


In [8]:
warnings.filterwarnings("ignore")

trimmed_auroc_ll_sets = iteratively_trim_feature_set(
    train_cohort_ll,
    logreg_ll_roc_candidate_set,
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='ll_v2',
    # n_loops = 3,
    random_state_list=[42],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auroc_ll_set_logreg_scores,
    original_feature_set_svc_scores=full_auroc_ll_set_svc_scores,
    max_iter_svc=1000,
    allow_overwriting_files=True,
    subfolder_name=''
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
mbp_min (loop 1 of 1, feature 1 of 37)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.9004
Temp mean logreg AUC: 0.8956
mean of diffs = -0.0048
sd of diffs = 0.0107
t = -1.4181
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
MCV_mean (loop 1 of 1, feature 2 of 37)
Total elapsed time: 0.12 minutes.
Loop elapsed time: 0.12 minutes.
Original mean logreg AUC: 0.9004
Temp mean logreg AUC: 0.8920
mean of diffs = -0.0083
sd of diffs = 0.0112
t = -2.3496
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
resp_rate_mean (loop 1 of 1, feature 3 of 37)
Total elapsed time: 0.20 minutes.
Loop elapsed time: 0.20 minutes.
Original mean logreg AUC: 0.9004
Temp mean logreg AUC: 0.8929
mean of diffs = -0.0075
sd of diffs = 0.0105
t = -2.2367
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
mbp_median (loop 1 of 1, feature 4 of 37)
Total elapsed time: 0.29 minutes.
Loop elapsed time: 0.29 minutes.
Origi

In [9]:
for i in range(len(trimmed_auroc_ll_sets)):
    print('~'*15 + '\nSet %d\n' % i + '~'*15)
    for feature in trimmed_auroc_ll_sets[i]:
        print(feature)
    print('\n')

~~~~~~~~~~~~~~~
Set 0
~~~~~~~~~~~~~~~
mbp_min
MCV_mean
resp_rate_mean
mbp_median
POTASSIUM_mean
FERRITIN_median
BUN_min
map_mean
PT_mean
CRP_min
RDW_min
MAGNESIUM_min
POTASSIUM_min
pox_mean
pulse_median
LACTATE_mean
CALCIUM_ION_min
ALT_mean
dbp_max
mbp_max
ALBUMIN_median
CREATININE_median
BUN_median
resp_rate_min
BUN_mean
sbp_median
cardiac_arrest
GLUCOSE_min
NEUTRO_PCT_mean
sbp_mean
PROCALCITONIN_mean
fio2_max
temp_min
temp_median
AST_mean
CREATININE_mean
fio2_mean




In [10]:
with open('pickle/trimmed_feature_sets/ll_auroc_svc_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_ll_sets, outfile)

### AUPRC

In [11]:
with open('pickle/candidate_features/svc_latest_lab_auprc_candidate_features.pickle', 'rb') as infile:
    final_ll_prc_candidate_set = pickle.load(infile)

In [12]:
len(final_ll_prc_candidate_set)

42

In [13]:
full_auprc_ll_set_logreg_scores, full_auprc_ll_set_svc_scores = get_or_load_initial_scores(
    train_cohort_ll,
    final_ll_prc_candidate_set,
    'average_precision',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/ll/auprc.pickle',
    get_svc_scores=True
)

Loading full logistic regression scores...
Loading full SVC scores...


In [14]:
# full_auprc_ll_set_logreg_scores = train_logreg_cv_from_feature_set(
#     train_cohort_ll, 
#     final_ll_prc_candidate_set, 
#     scoring_metric='average_precision',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343
# )

# full_auprc_ll_set_svc_scores = train_svc_cv_from_feature_set(
#     train_cohort_ll, 
#     final_ll_prc_candidate_set, 
#     scoring_metric='average_precision',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343,
#     max_iter=2500
# )

In [15]:
print('mean logreg score: %.4f' % np.mean(full_auprc_ll_set_logreg_scores))
print('mean SVC score: %.4f' % np.mean(full_auprc_ll_set_svc_scores))

mean logreg score: 0.3545
mean SVC score: 0.1760


In [16]:
trimmed_auprc_ll_sets = iteratively_trim_feature_set(
    train_cohort_ll,
    final_ll_prc_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    random_state_list = [42],
    pval_threshold = 0.05,
    imp_method_str='ll',
    final_feature_set_list = [],
    # random_state=343,
    original_feature_set_logreg_scores=full_auprc_ll_set_logreg_scores,
    original_feature_set_svc_scores=full_auprc_ll_set_svc_scores,
    max_iter_svc=2500
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
PHOSPHORUS_median (loop 1 of 1, feature 1 of 42)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.3545
Temp mean logreg AUC: 0.3502
mean of diffs = -0.0044
sd of diffs = 0.0363
t = -0.3789
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
mbp_median (loop 1 of 1, feature 2 of 42)
Total elapsed time: 0.15 minutes.
Loop elapsed time: 0.15 minutes.
Original mean logreg AUC: 0.3545
Temp mean logreg AUC: 0.3496
mean of diffs = -0.0049
sd of diffs = 0.0354
t = -0.4376
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
PCO2_median (loop 1 of 1, feature 3 of 42)
Total elapsed time: 0.30 minutes.
Loop elapsed time: 0.30 minutes.
Original mean logreg AUC: 0.3545
Temp mean logreg AUC: 0.3503
mean of diffs = -0.0042
sd of diffs = 0.0363
t = -0.3654
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
MAGNESIUM_max (loop 1 of 1, feature 4 of 42)
Total elapsed time: 0.44 minutes.
Loop elapsed time: 0.44 mi

In [17]:
for i in range(len(trimmed_auprc_ll_sets)):
    print('~'*15 + '\nSet %d\n' % i + '~'*15)
    for feature in trimmed_auprc_ll_sets[i]:
        print(feature)
    print('\n')

~~~~~~~~~~~~~~~
Set 0
~~~~~~~~~~~~~~~
pulse_mean
sbp_mean
pulse_median
PHOSPHORUS_median
BUN_median
MCV_mean
mbp_median
PCO2_median
map_mean
MAGNESIUM_max
PLTS_mean
resp_rate_mean
BUN_min
PHOSPHORUS_mean
temp_min
BASE_EXC_median
BASE_EXC_mean
PCO2_mean
CALCIUM_ION_max
temp_median
ALC_median
PH_mean
BICARBONATE_min
dbp_max
CALCIUM_ION_min
NEUTRO_PCT_mean
LACTATE_mean
NEUTRO_PCT_median
pulse_max
GLUCOSE_min
CREATININE_median
sbp_median
PO2_median
mbp_min
mbp_max
pox_min
ALBUMIN_mean
BUN_mean
MAGNESIUM_mean
CREATININE_mean
map_min
pox_mean




In [18]:
with open('pickle/trimmed_feature_sets/trimmed_auprc_ll_svc_sets.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_ll_sets, outfile)

In [19]:
# with open('pickle/trimmed_feature_sets/v2/trimmed_auprc_ll_svc_sets.pickle', 'wb') as outfile:
#     pickle.dump(trimmed_auprc_ll_sets, outfile)

## Median imputation

In [20]:
SVC_SAMPLE_PROPORTION = 0.25

### AUROC

In [21]:
with open('pickle/candidate_features/svc_median_imp_auproc_candidate_features.pickle', 'rb') as infile:
    final_med_roc_candidate_set = pickle.load(infile)

In [22]:
full_auroc_med_set_logreg_scores, full_auroc_med_set_svc_scores = get_or_load_initial_scores(
    train_cohort_med,
    final_med_roc_candidate_set,
    'roc_auc',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/med/auroc.pickle',
    get_svc_scores=True,
    svc_sample_proportion=SVC_SAMPLE_PROPORTION,
    svc_sample_random_state=343
)

Loading full logistic regression scores...
Loading full SVC scores...


In [23]:
warnings.filterwarnings("ignore")

trimmed_auroc_med_sets = iteratively_trim_feature_set(
    train_cohort_med,
    final_med_roc_candidate_set,
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='median',
    # n_loops = 3,
    random_state_list=[42],
    pval_threshold = 0.1,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auroc_med_set_logreg_scores,
    original_feature_set_svc_scores=full_auroc_med_set_svc_scores,
    max_iter_svc=2500,
    svc_sample_proportion=SVC_SAMPLE_PROPORTION,
    svc_sample_random_state=343
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
pulse_max (loop 1 of 1, feature 1 of 44)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.8400
Temp mean logreg AUC: 0.8343
mean of diffs = -0.0057
sd of diffs = 0.0171
t = -1.0626
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
MCV_min (loop 1 of 1, feature 2 of 44)
Total elapsed time: 0.11 minutes.
Loop elapsed time: 0.11 minutes.
Original mean logreg AUC: 0.8400
Temp mean logreg AUC: 0.8362
mean of diffs = -0.0039
sd of diffs = 0.0160
t = -0.7658
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
BANDS_PCT_mean (loop 1 of 1, feature 3 of 44)
Total elapsed time: 0.21 minutes.
Loop elapsed time: 0.21 minutes.
Original mean logreg AUC: 0.8400
Temp mean logreg AUC: 0.8362
mean of diffs = -0.0038
sd of diffs = 0.0160
t = -0.7536
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
pulse_median (loop 1 of 1, feature 4 of 44)
Total elapsed time: 0.33 minutes.
Loop elapsed time: 0.33 minutes.
Or

In [24]:
with open('pickle/trimmed_feature_sets/med_auroc_svc_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_med_sets, outfile)

### AUPRC

In [25]:
with open('pickle/candidate_features/svc_median_auprc_candidate_feature_set.pickle', 'rb') as infile:
    final_med_prc_candidate_set = pickle.load(infile)

In [26]:
# full_auprc_med_set_logreg_scores = train_logreg_cv_from_feature_set(
#     train_cohort_med, 
#     final_med_prc_candidate_set, 
#     scoring_metric='average_precision',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343
# )

# full_auprc_med_set_svc_scores = train_svc_cv_from_feature_set(
#     train_cohort_med, 
#     final_med_prc_candidate_set, 
#     scoring_metric='average_precision',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343,
#     max_iter=2500
# )

In [27]:
# with open('pickle/trimmed_feature_sets/median_average_precision_feature_dicts/all_candidate_features_logreg_scores.pickle', 'wb') as outfile:
#     pickle.dump(full_auprc_med_set_logreg_scores, outfile)
    
# with open('pickle/trimmed_feature_sets/median_average_precision_feature_dicts/all_candidate_features_svc_scores.pickle', 'wb') as outfile:
#     pickle.dump(full_auprc_med_set_svc_scores, outfile)

In [28]:
full_auprc_med_set_logreg_scores, full_auprc_med_set_svc_scores = get_or_load_initial_scores(
    train_cohort_med,
    final_med_prc_candidate_set,
    'average_precision',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/med/auprc.pickle',
    get_svc_scores=True,
    svc_sample_proportion=SVC_SAMPLE_PROPORTION,
    svc_sample_random_state=343
)

Loading full logistic regression scores...
Loading full SVC scores...


In [29]:
warnings.filterwarnings("ignore")

trimmed_auprc_med_sets = iteratively_trim_feature_set(
    train_cohort_med,
    final_med_prc_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='median_v2',
    # n_loops = 3,
    random_state_list=[42],
    pval_threshold = 0.10,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auprc_med_set_logreg_scores,
    original_feature_set_svc_scores=full_auprc_med_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    svc_sample_proportion=SVC_SAMPLE_PROPORTION,
    svc_sample_random_state=343
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
ALBUMIN_min (loop 1 of 1, feature 1 of 56)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.2267
Temp mean logreg AUC: 0.2281
mean of diffs = 0.0014
sd of diffs = 0.0323
t = 0.1346
Logistic regression model not significantly worse after removing ALBUMIN_min.
Logistic regression scores (new | old | new - old):
0.2693 | 0.2056 | 0.0636
0.2723 | 0.2549 | 0.0174
0.2055 | 0.2074 | -0.0019
0.2301 | 0.2464 | -0.0163
0.1942 | 0.2482 | -0.0541
0.2160 | 0.2252 | -0.0092
0.1995 | 0.2186 | -0.0191
0.2245 | 0.2347 | -0.0102
0.2030 | 0.2160 | -0.0130
0.2667 | 0.2102 | 0.0565

Assessing SVC performance...
Original mean SVC AUC: 0.1382
Temp mean SVC AUC: 0.0538
mean of diffs = -0.0845
sd of diffs = 0.0850
t = -3.1430
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
MCV_mean (loop 1 of 1, feature 2 of 56)
Total elapsed time: 1.16 minutes.
Loop elapsed time: 1.16 minutes.
Original mean log

/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.8/site-packages/sklearn/svm/_bas

Original mean SVC AUC: 0.1382
Temp mean SVC AUC: 0.0264
mean of diffs = -0.1118
sd of diffs = 0.0706
t = -5.0070
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
sbp_mean (loop 1 of 1, feature 17 of 56)
Total elapsed time: 9.49 minutes.
Loop elapsed time: 9.49 minutes.
Original mean logreg AUC: 0.2304
Temp mean logreg AUC: 0.2281
mean of diffs = -0.0023
sd of diffs = 0.0025
t = -2.8955
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
LACTATE_mean (loop 1 of 1, feature 18 of 56)
Total elapsed time: 9.65 minutes.
Loop elapsed time: 9.65 minutes.
Original mean logreg AUC: 0.2304
Temp mean logreg AUC: 0.2260
mean of diffs = -0.0044
sd of diffs = 0.0060
t = -2.3103
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
CHLORIDE_max (loop 1 of 1, feature 19 of 56)
Total elapsed time: 9.81 minutes.
Loop elapsed time: 9.81 minutes.
Original mean logreg AUC: 0.2304
Temp mean logreg AUC: 0.2271
mean of diffs = -0.0033
sd of diffs = 0.0030
t = -3.4910
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
map_mean (loop 1 of 1, feature

In [30]:
with open('pickle/trimmed_feature_sets/med_auprc_svc_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_med_sets, outfile)

# Trim features sets (logistic regression only)

## Latest lab imputation

### AUROC

In [31]:
with open('pickle/candidate_features/logreg_latest_lab_auroc_candidate_feature_set.pickle', 'rb') as infile:
    logreg_ll_roc_candidate_set = pickle.load(infile)

In [32]:
full_auroc_ll_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_ll,
    logreg_ll_roc_candidate_set,
    'roc_auc',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/ll/auroc.pickle',
    get_svc_scores=False
)

Loading full logistic regression scores...


In [33]:
warnings.filterwarnings("ignore")

trimmed_auroc_ll_sets_logreg_only = iteratively_trim_feature_set(
    train_cohort_ll,
    logreg_ll_roc_candidate_set,
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='ll_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auroc_ll_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    subfolder_name='',
    logreg_only=True
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
CREATININE_median (loop 1 of 3, feature 1 of 36)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.9004
Temp mean logreg AUC: 0.8952
mean of diffs = -0.0052
sd of diffs = 0.0193
t = -0.8503
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
LACTATE_mean (loop 1 of 3, feature 2 of 36)
Total elapsed time: 0.11 minutes.
Loop elapsed time: 0.11 minutes.
Original mean logreg AUC: 0.9004
Temp mean logreg AUC: 0.8951
mean of diffs = -0.0053
sd of diffs = 0.0183
t = -0.9120
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
temp_min (loop 1 of 3, feature 3 of 36)
Total elapsed time: 0.22 minutes.
Loop elapsed time: 0.22 minutes.
Original mean logreg AUC: 0.9004
Temp mean logreg AUC: 0.8951
mean of diffs = -0.0053
sd of diffs = 0.0195
t = -0.8572
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
NEUTRO_PCT_mean (loop 1 of 3, feature 4 of 36)
Total elapsed time: 0.33 minutes.
Loop elapsed time: 0.33 m

In [34]:
with open('pickle/trimmed_feature_sets/ll_auroc_logreg_only_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_ll_sets_logreg_only, outfile)

#### 10 iter

In [ ]:
# full_auroc_ll_set_logreg_scores = get_or_load_initial_scores(
#     train_cohort_ll,
#     logreg_ll_roc_candidate_set,
#     'roc_auc',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343,
#     max_iter_svc=2500,
#     logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/ll/auroc.pickle',
#     get_svc_scores=False
# )

In [ ]:
# warnings.filterwarnings("ignore")

# trimmed_auroc_ll_sets_logreg_only_10_iter = iteratively_trim_feature_set(
#     train_cohort_ll,
#     logreg_ll_roc_candidate_set,
#     'roc_auc', # 'roc_auc' or 'average_precision'
#     imp_method_str='ll_logreg_only',
#     # n_loops = 3,
#     random_state_list=[0, 42, 343, 52, 7345, 1, 9, 12, 100, 1000],
#     pval_threshold = 0.05,
#     final_feature_set_list = [],
#     random_state=343,
#     original_feature_set_logreg_scores=full_auroc_ll_set_logreg_scores,
#     # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
#     max_iter_svc=2500,
#     allow_overwriting_files=True,
#     subfolder_name='',
#     logreg_only=True
# )

# with open('pickle/trimmed_feature_sets/ll_auroc_logreg_only_10_iter.pickle', 'wb') as outfile:
#     pickle.dump(trimmed_auroc_ll_sets_logreg_only_10_iter, outfile)

### AUPRC

In [ ]:
with open('pickle/candidate_features/logreg_latest_lab_auprc_candidate_feature_set.pickle', 'rb') as infile:
    logreg_ll_prc_candidate_set = pickle.load(infile)

In [ ]:
full_auprc_ll_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_ll,
    logreg_ll_prc_candidate_set,
    'average_precision',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/ll/auprc.pickle',
    get_svc_scores=False
)

In [ ]:
warnings.filterwarnings("ignore")

trimmed_auprc_ll_sets_logreg_only = iteratively_trim_feature_set(
    train_cohort_ll,
    logreg_ll_prc_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='ll_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auprc_ll_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=False,
    subfolder_name='v2',
    logreg_only=True
)

In [ ]:
# with open('pickle/trimmed_feature_sets/v2/logreg_only/trimmed_auprc_ll_v2_logreg_only_sets.pickle', 'wb') as outfile:
#     pickle.dump(trimmed_auprc_ll_sets_v2_logreg_only, outfile)

with open('pickle/trimmed_feature_sets/ll_auprc_logreg_only_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_ll_sets_logreg_only, outfile)

#### 10 iter

In [ ]:
full_auprc_ll_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_ll,
    logreg_ll_prc_candidate_set,
    'average_precision',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/ll/auprc.pickle',
    get_svc_scores=False
)

In [ ]:
warnings.filterwarnings("ignore")

trimmed_auroc_ll_sets_logreg_only_10_iter = iteratively_trim_feature_set(
    train_cohort_ll,
    logreg_ll_roc_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='ll_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343, 52, 7345, 1, 9, 12, 100, 1000],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auprc_ll_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    subfolder_name='',
    logreg_only=True
)



In [ ]:
with open('pickle/trimmed_feature_sets/ll_auprc_logreg_only_10_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_ll_sets_logreg_only_10_iter, outfile)

## Median imputation

### AUROC

In [ ]:
with open('pickle/candidate_features/logreg_median_auroc_candidate_feature_set.pickle', 'rb') as infile:
    roc_med_candidate_set = pickle.load(infile)

In [ ]:
full_auroc_med_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_med,
    roc_med_candidate_set,
    'roc_auc',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/med/auroc.pickle',
    get_svc_scores=False
)

In [ ]:
# print('\n'.join(roc_med_v2_candidate_set))

In [ ]:
warnings.filterwarnings("ignore")

trimmed_auroc_med_sets_logreg_only = iteratively_trim_feature_set(
    train_cohort_med,
    roc_med_candidate_set,
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='med_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auroc_med_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=False,
    subfolder_name='',
    logreg_only=True
)

In [ ]:
with open('pickle/trimmed_feature_sets/med_auroc_logreg_only_3_iter.pickle.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_med_sets_logreg_only, outfile)

#### 10 iter

In [ ]:
warnings.filterwarnings("ignore")

trimmed_auroc_med_sets_logreg_only_10_iter = iteratively_trim_feature_set(
    train_cohort_med,
    roc_med_candidate_set,
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='med_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343, 52, 7345, 1, 9, 12, 100, 1000],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auroc_med_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    subfolder_name='',
    logreg_only=True
)

with open('pickle/trimmed_feature_sets/med_auroc_logreg_only_10_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_med_sets_logreg_only_10_iter, outfile)

### AUPRC

In [ ]:
with open('pickle/candidate_features/logreg_median_auprc_candidate_feature_set_v2.pickle', 'rb') as infile:
    prc_med_candidate_set = pickle.load(infile)

In [ ]:
full_auprc_med_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_med,
    prc_med_candidate_set,
    'average_precision',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/all_features_initial_scores/logreg/med/auprc.pickle',
    # v2/med_v2_average_precision_feature_dicts/all_candidate_features_logreg_scores.pickle',
    get_svc_scores=False
)

In [ ]:
warnings.filterwarnings("ignore")

trimmed_auprc_med_sets_logreg_only = iteratively_trim_feature_set(
    train_cohort_med,
    prc_med_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='med_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auprc_med_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=False,
    subfolder_name='',
    logreg_only=True
)

In [ ]:
with open('pickle/trimmed_feature_sets/med_auprc_logreg_only_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_med_sets_logreg_only, outfile)

#### 10 iter

In [ ]:
warnings.filterwarnings("ignore")

trimmed_auprc_med_sets_logreg_only_10_iter = iteratively_trim_feature_set(
    train_cohort_med,
    roc_med_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='med_logreg_only',
    # n_loops = 3,
    random_state_list=[0, 42, 343, 52, 7345, 1, 9, 12, 100, 1000],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auprc_med_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    subfolder_name='',
    logreg_only=True
)

with open('pickle/trimmed_feature_sets/med_auprc_logreg_only_10_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_med_sets_logreg_only_10_iter, outfile)

# Trim features sets (logistic regression only; 10 trimming iterations each)

In [1]:
# random_states_10_iter = [846, 626, 2024, 1, 100, 343 * 42, 343 * 24, 0, 42, 343]

## Latest lab imputation

### AUROC

In [ ]:
# if os.path.isfile('pickle/trimmed_feature_sets/v2/ll_v2_roc_auc_feature_dicts/all_candidate_features_logreg_scores.pickle'):
#     print('Loading full logistic regression scores...')
#     with open('pickle/trimmed_feature_sets/v2/ll_v2_roc_auc_feature_dicts/all_candidate_features_logreg_scores.pickle', 'rb') as infile:
#         full_auroc_ll_set_logreg_scores = pickle.load(infile)
# else:
#     print('Calculating full logistic regression scores...')
#     full_auroc_ll_set_logreg_scores = train_logreg_cv_from_feature_set(
#         train_cohort_ll, 
#         final_ll_roc_v2_candidate_set, 
#         scoring_metric='roc_auc',
#         n_splits=10,
#         n_jobs=10,
#         random_state=343
#     )
    
#     with open('pickle/trimmed_feature_sets/v2/ll_v2_roc_auc_feature_dicts/all_candidate_features_logreg_scores.pickle', 'wb') as outfile:
#         pickle.dump(full_auroc_ll_set_logreg_scores, outfile)

In [ ]:
# full_auroc_ll_set_logreg_scores = get_or_load_initial_scores(
#     train_cohort_ll,
#     final_ll_roc_v2_candidate_set,
#     'roc_auc',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343,
#     max_iter_svc=2500,
#     logreg_score_file_path='pickle/trimmed_feature_sets/v2/ll_v2_roc_auc_feature_dicts/all_candidate_features_logreg_scores.pickle',
#     get_svc_scores=False
# )

In [ ]:
# warnings.filterwarnings("ignore")

# trimmed_auroc_ll_sets_v2_logreg_only_10_iter = iteratively_trim_feature_set(
#     train_cohort_ll,
#     final_ll_roc_v2_candidate_set,
#     'roc_auc', # 'roc_auc' or 'average_precision'
#     imp_method_str='ll_v2_logreg_only_10_iter',
#     # n_loops = 3,
#     random_state_list=random_states_10_iter,
#     pval_threshold = 0.05,
#     final_feature_set_list = [],
#     random_state=343,
#     original_feature_set_logreg_scores=full_auroc_ll_set_logreg_scores,
#     # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
#     max_iter_svc=2500,
#     allow_overwriting_files=True,
#     subfolder_name='',
#     logreg_only=True
# )

In [ ]:
# with open('pickle/trimmed_feature_sets/v2/logreg_only/10_iter/trimmed_auroc_ll_v2_logreg_only_sets_10_iter.pickle', 'wb') as outfile:
#     pickle.dump(trimmed_auroc_ll_sets_v2_logreg_only_10_iter, outfile)random_states_10_iter

### AUPRC

In [15]:
# with open('pickle/candidate_features/svc_latest_lab_auprc_candidate_features_v2.pickle', 'rb') as infile:
#     final_ll_prc_candidate_set = pickle.load(infile)

In [16]:
# full_auprc_ll_set_logreg_scores = get_or_load_initial_scores(
#     train_cohort_ll,
#     final_ll_prc_candidate_set,
#     'average_precision',
#     n_splits=10,
#     n_jobs=10,
#     random_state=343,
#     max_iter_svc=2500,
#     logreg_score_file_path='pickle/trimmed_feature_sets/v2/ll_v2_average_precision_feature_dicts/all_candidate_features_logreg_scores.pickle',
#     get_svc_scores=False
# )

Loading full logistic regression scores...


In [19]:
# warnings.filterwarnings("ignore")

# trimmed_auprc_ll_sets_v2_logreg_only_10_iter = iteratively_trim_feature_set(
#     train_cohort_ll,
#     final_ll_prc_candidate_set,
#     'average_precision', # 'roc_auc' or 'average_precision'
#     imp_method_str='ll_v2_logreg_only_10_iter',
#     # n_loops = 3,
#     random_state_list=random_states_10_iter,
#     pval_threshold = 0.05,
#     final_feature_set_list = [],
#     random_state=343,
#     original_feature_set_logreg_scores=full_auprc_ll_set_logreg_scores,
#     # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
#     max_iter_svc=2500,
#     allow_overwriting_files=False,
#     subfolder_name='v2',
#     logreg_only=True
# )

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
BILIRUBIN_DIR_min (loop 1 of 10, feature 1 of 43)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.2322
Temp mean logreg AUC: 0.2575
mean of diffs = 0.0253
sd of diffs = 0.0758
t = 1.0558
Logistic regression model not significantly worse after removing BILIRUBIN_DIR_min.
Logistic regression scores (new | old | new - old):
0.2766 | 0.1903 | 0.0863
0.2806 | 0.1742 | 0.1065
0.3302 | 0.2472 | 0.0830
0.2428 | 0.3095 | -0.0668
0.2108 | 0.2983 | -0.0875
0.2833 | 0.1780 | 0.1053
0.1850 | 0.2869 | -0.1019
0.2208 | 0.2344 | -0.0136
0.2805 | 0.2032 | 0.0774
0.2639 | 0.1996 | 0.0643

Removing BILIRUBIN_DIR_min from current feature set.
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
PLTS_mean (loop 1 of 10, feature 2 of 43)
Total elapsed time: 0.07 minutes.
Loop elapsed time: 0.07 minutes.
Original mean logreg AUC: 0.2575
Temp mean logreg AUC: 0.2666
mean of diffs = 0.0091
sd of dif

In [20]:
with open('pickle/trimmed_feature_sets/v2/logreg_only/10_iter/trimmed_auprc_ll_v2_logreg_only_sets_10_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_ll_sets_v2_logreg_only_10_iter, outfile)

## Median imputation

### AUROC

In [21]:
with open('pickle/candidate_features/svc_median_imp_auroc_candidate_features_v2.pickle', 'rb') as infile:
    roc_med_v2_candidate_set = pickle.load(infile)

In [22]:
full_auroc_med_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_med,
    roc_med_v2_candidate_set,
    'roc_auc',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/v2/med_v2_roc_auc_feature_dicts/all_candidate_features_logreg_scores.pickle',
    get_svc_scores=False
)

Loading full logistic regression scores...


In [ ]:
# print('\n'.join(roc_med_v2_candidate_set))

In [23]:
warnings.filterwarnings("ignore")

trimmed_auroc_med_sets_v2_logreg_only_10_iter = iteratively_trim_feature_set(
    train_cohort_med,
    roc_med_v2_candidate_set,
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='med_v2_logreg_only_10_iter',
    # n_loops = 3,
    random_state_list=random_states_10_iter,
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auroc_med_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=False,
    subfolder_name='v2',
    logreg_only=True
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
PH_min (loop 1 of 10, feature 1 of 40)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.8521
Temp mean logreg AUC: 0.8185
mean of diffs = -0.0337
sd of diffs = 0.0421
t = -2.5258
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
ALT_mean (loop 1 of 10, feature 2 of 40)
Total elapsed time: 0.05 minutes.
Loop elapsed time: 0.05 minutes.
Original mean logreg AUC: 0.8521
Temp mean logreg AUC: 0.8286
mean of diffs = -0.0235
sd of diffs = 0.0420
t = -1.7745
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
pox_mean (loop 1 of 10, feature 3 of 40)
Total elapsed time: 0.11 minutes.
Loop elapsed time: 0.11 minutes.
Original mean logreg AUC: 0.8521
Temp mean logreg AUC: 0.8457
mean of diffs = -0.0064
sd of diffs = 0.0380
t = -0.5326
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
CALCIUM_ION_min (loop 1 of 10, feature 4 of 40)
Total elapsed time: 0.18 minutes.
Loop elapsed time: 0.18 minutes.
Ori

In [24]:
with open('pickle/trimmed_feature_sets/v2/logreg_only/10_iter/trimmed_auroc_med_v2_logreg_only_sets_10_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_med_sets_v2_logreg_only_10_iter, outfile)

### AUPRC

In [25]:
with open('pickle/candidate_features/svc_med_auprc_candidate_features_v2.pickle', 'rb') as infile:
    prc_med_v2_candidate_set = pickle.load(infile)

In [26]:
full_auprc_med_set_logreg_scores = get_or_load_initial_scores(
    train_cohort_med,
    prc_med_v2_candidate_set,
    'average_precision',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    logreg_score_file_path='pickle/trimmed_feature_sets/v2/med_v2_average_precision_feature_dicts/all_candidate_features_logreg_scores.pickle',
    get_svc_scores=False
)

Loading full logistic regression scores...


In [27]:
warnings.filterwarnings("ignore")

trimmed_auprc_med_sets_v2_logreg_only_10_iter = iteratively_trim_feature_set(
    train_cohort_med,
    prc_med_v2_candidate_set,
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='med_v2_logreg_only_10_iter',
    # n_loops = 3,
    random_state_list=random_states_10_iter,
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=full_auprc_med_set_logreg_scores,
    # original_feature_set_svc_scores=full_auroc_svc_set_svc_scores,
    max_iter_svc=2500,
    allow_overwriting_files=False,
    subfolder_name='v2',
    logreg_only=True
)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
BILIRUBIN_DIR_min (loop 1 of 10, feature 1 of 43)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.2180
Temp mean logreg AUC: 0.2575
mean of diffs = 0.0395
sd of diffs = 0.0617
t = 2.0213
Logistic regression model not significantly worse after removing BILIRUBIN_DIR_min.
Logistic regression scores (new | old | new - old):
0.2766 | 0.1903 | 0.0863
0.2806 | 0.1737 | 0.1069
0.3302 | 0.2471 | 0.0831
0.2428 | 0.2206 | 0.0222
0.2108 | 0.2897 | -0.0789
0.2833 | 0.1779 | 0.1054
0.1850 | 0.2443 | -0.0594
0.2208 | 0.2340 | -0.0132
0.2805 | 0.2027 | 0.0779
0.2639 | 0.1996 | 0.0644

Removing BILIRUBIN_DIR_min from current feature set.
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
PLTS_mean (loop 1 of 10, feature 2 of 43)
Total elapsed time: 0.04 minutes.
Loop elapsed time: 0.04 minutes.
Original mean logreg AUC: 0.2575
Temp mean logreg AUC: 0.2666
mean of diffs = 0.0091
sd of diff

In [28]:
with open('pickle/trimmed_feature_sets/v2/logreg_only/10_iter/trimmed_auprc_med_v2_logreg_only_sets_10_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_med_sets_v2_logreg_only_10_iter, outfile)